In [1]:
import sqlite3
import pandas as pd
import os
import re
import time
from tqdm import tqdm

### Variables from conf file

In [2]:
# database file path
DB_FILE = "../../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"

# transaction table to update with status
TRANSACTION_TABLE = "transaction_row"

# column to show morph-syntax errors
EKILEX_COL = "ekilex_tag"

# DIRs for ekilex tag files
DIRECTORY_OBL =  "../../../../v04_verb-case_pattern/base_data/v05_wordlists_obl"
DIRECTORY_ADVMOD =  "../../../../v04_verb-case_pattern/base_data/v05_wordlists_advmod"

### Save word and semantic type to dict

In [3]:
def word_semtype_fun(directory_str):
    word_semtype = [] # tuples of semtype and word, i.e. (amount, aegsamini)
    
    directory = os.fsencode(directory_str)
    
    for file in os.listdir(directory):
        file_name = os.fsdecode(file)
        filepath = directory_str + '/' + file_name
        semtype = re.findall(r"^(?:adv_)?(.+?)\.[^.]+$", file_name)
        with open(filepath, "r", encoding="utf-8") as f:
            words = f.read().splitlines()
            for word in words:
                word_semtype.append((semtype[0], word))  
    return word_semtype

In [4]:
word_semtype_obl = word_semtype_fun(DIRECTORY_OBL)
word_semtype_adv = word_semtype_fun(DIRECTORY_ADVMOD)

### Add semantic types to lemmas in the database


In [5]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [6]:
def semtype_to_db(deprel, table_name, semtypes, db_file, tag_col):
    
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    
    # Step 1: add new column to database table
    if not column_exists(cursor, table_name, tag_col):
        cursor.execute("ALTER TABLE " + table_name + f" ADD COLUMN {tag_col} TEXT")

    # Step 2: add word + semantic tag to temporary table
    cursor.execute(f"CREATE TEMP TABLE IF NOT EXISTS temp_updates (lemma TEXT PRIMARY KEY, {tag_col} TEXT)")
    cursor.executemany(f"INSERT INTO temp_updates ({tag_col}, lemma) VALUES (?, ?)", semtypes)

    # Step 3: Add temporary table info to database table
    #ps, pronouns are excluded for spatial obliques
    cursor.execute(f"""
        UPDATE {table_name}
        SET {tag_col} = (SELECT {tag_col} FROM temp_updates WHERE temp_updates.lemma = {table_name}.lemma)
        WHERE pos != 'P' 
                AND EXISTS (SELECT 1 FROM temp_updates WHERE temp_updates.lemma = {table_name}.lemma)
                AND deprel='{deprel}'
    """)

    conn.commit()
    conn.close()

In [7]:
semtype_to_db("obl", TRANSACTION_TABLE, word_semtype_obl, DB_FILE, EKILEX_COL)

In [8]:
semtype_to_db('advmod', TRANSACTION_TABLE, word_semtype_adv, DB_FILE, EKILEX_COL)

### Check the results (not part of final workflow)

### Connect to database

In [9]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

In [10]:
query = f"SELECT * FROM transaction_row limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,status,ekilex_tag
0,1,2,3,-1,obl,lõpus,lõpp,"com,in,sg",None,S,,None
1,2,2,5,1,nsubj,Türi,Türi,"gen,prop,sg",None,S,syntax-morph conflict,None
2,3,2,6,2,obl,1.,1.,"<?>,ord,roman",None,N,,None
3,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S,syntax-morph conflict,None
4,5,3,9,-2,nsubj,kidramees,kidramees,"com,nom,sg",None,S,,None
5,6,3,10,-1,aux,ei,ei,"aux,neg",None,V,,None
6,7,3,12,1,obl,keeltele,keel,"all,com,pl",None,S,,None
7,8,3,13,2,compound:prt,pihta,pihta,,None,D,,None
8,9,4,4,-2,nsubj,solist,solist,"com,nom,sg",None,S,,None
9,10,4,5,-1,aux,ei,ei,"aux,neg",None,V,,None


In [11]:
query = f"SELECT * FROM transaction_row where deprel='obl' and ekilex_tag is not null limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,status,ekilex_tag
0,56,31,3,2,obl,diskorina,diskor,"com,es,sg",None,S,,alive
1,59,33,12,1,obl,rahvale,rahvas,"all,com,sg",None,S,,alive
2,119,74,5,1,obl,toidupoest,toidupood,"com,el,sg",None,S,,location
3,131,85,9,1,obl,bändidele,bänd,"all,com,pl",None,S,,alive
4,148,96,5,2,obl,inimestest,inimene,"com,el,pl",None,S,,alive
5,170,109,13,-2,obl,peost,pidu,"com,el,sg",None,S,,event
6,196,135,5,1,obl,kontserdile,kontsert,"all,com,sg",None,S,,event
7,248,172,4,1,obl,pedagoogikaülikoolis,pedagoogikaülikool,"com,in,sg",None,S,,location
8,281,190,13,1,obl,kevadel,kevad,"ad,com,sg",None,S,,time
9,323,219,25,2,obl,alkoholismist,alkoholism,"com,el,sg",None,S,,state


In [12]:
query = f"SELECT * FROM transaction_row where deprel='advmod' and ekilex_tag is not null limit 10"

res = pd.read_sql(query, conn)
res

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,status,ekilex_tag
0,18,10,2,-1,advmod,sageli,sageli,,None,D,,time
1,31,19,4,1,advmod,kaheksaselt,kaheksaselt,,None,D,,time
2,45,28,1,-1,advmod,Tasapisi,tasapisi,,None,D,,manner
3,46,28,3,1,advmod,aga,aga,"crd,sub",None,J,syntax-morph conflict,general
4,62,35,1,-1,advmod,Hiljem,hiljem,,None,D,,time
5,65,36,4,2,advmod,meeleldi,meeleldi,,None,D,,manner
6,67,37,1,-2,advmod,Pidevalt,pidevalt,,None,D,,time
7,71,40,1,-2,advmod,Kas,kas,,None,D,,general
8,135,90,2,-1,advmod,pigem,pigem,,None,D,,general
9,159,101,4,-2,advmod,mistõttu,mistõttu,,None,D,,general


In [13]:
conn.close()